# Introduction to Machine Learning — Class 5
## Comparing Models: Naive Bayes, Decision Trees & Linear SVM

**Input dataset:** `adult_classification_v1.csv`  
**Problem:** Binary classification  

---

### Learning goals
- Compare different classification models using the same data
- Understand model assumptions and inductive bias
- Analyse interpretability vs flexibility
- Reflect on how model choice affects decisions

> **Key idea:** Changing the model changes the assumptions we make about the data.

<div style="margin-left: 2em; font-size: 0.9em; font-style: italic;">
An AI language model (ChatGPT by OpenAI) was used to support the creation of practical class materials. 
All arguments, outputs, and final wording were critically reviewed, edited, and validated by the author 
prior to use with students.
</div>

## 0. Data context

In this session, you will reuse **exactly the same dataset** as in Session 4.

- Target: `income` (binary)
- Features: numerical, preprocessed beforehand

Nothing changes in the data.  
**Only the model changes.**


In [3]:
DATA_PATH = "adult_classification_v1.csv"

## 1. Imports


In [ ]:
# TODO: import required libraries
# numpy, pandas
# sklearn: train_test_split
# sklearn: GaussianNB, DecisionTreeClassifier, LinearSVC
# sklearn: accuracy_score

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


In [2]:
# Auto-check
import importlib
_required = ["numpy", "pandas", "sklearn"]
missing = [m for m in _required if importlib.util.find_spec(m) is None]
assert not missing, f"Missing packages: {missing}"
print("Auto-check passed.")


Auto-check passed.


## 2. Load the dataset

### Tasks
- Load the dataset into `df`
- Inspect class balance
- Evaluate if data scaling is needed

**Reflection**
- Did anything change since the previous session?


In [8]:
# TODO: load dataset
df = pd.read_csv(DATA_PATH)
df.head()
# check balance of target variable in %
print(df["income"].value_counts(normalize=True) * 100)

income
0    75.215603
1    24.784397
Name: proportion, dtype: float64


In [9]:
# Auto-check
import pandas as pd
assert "df" in globals(), "df not found"
assert "income" in df.columns
print("Auto-check passed.")


Auto-check passed.


## 3. Define X and y

### Task
- Separate features and target
- Split into training and validation sets

Keep the split consistent across models.


In [14]:
# TODO: define X, y and split
X = df.drop(columns=["income"])
y = df["income"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# check balance of target variable in train and val sets
print("Train set:")
print(y_train.value_counts(normalize=True) * 100)
print("Validation set:")
print(y_val.value_counts(normalize=True) * 100)


Train set:
income
0    75.216298
1    24.783702
Name: proportion, dtype: float64
Validation set:
income
0    75.212825
1    24.787175
Name: proportion, dtype: float64


## 4. Naive Bayes classifier

Naive Bayes makes a strong assumption:
**features are conditionally independent given the class**.

### Tasks
- Train a Gaussian Naive Bayes classifier
- Compute validation accuracy

**Reflection**
- Is the independence assumption realistic here?


In [27]:
# TODO: Naive Bayes, prediction and accuracy
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_val)
print(f"Naive Bayes accuracy: {accuracy_score(y_val, y_pred_nb):.4f}")
print(f"Naive Bayes precision: {precision_score(y_val, y_pred_nb):.4f}")
print(f"Naive Bayes recall: {recall_score(y_val, y_pred_nb):.4f}")

print(f"Naive Bayes F1-score: {f1_score(y_val, y_pred_nb):.4f}")

print("Naive Bayes Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_nb))


Naive Bayes accuracy: 0.7938
Naive Bayes precision: 0.6744
Naive Bayes recall: 0.3252
Naive Bayes F1-score: 0.4388
Naive Bayes Confusion Matrix:
[[6451  352]
 [1513  729]]


### 4b - NB with scalling

In [28]:
# Naive Bayes with scalling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
nb_scaled = GaussianNB()
nb_scaled.fit(X_train_scaled, y_train)
y_pred_nb_scaled = nb_scaled.predict(X_val_scaled)
print(f"Naive Bayes with scaling accuracy: {accuracy_score(y_val, y_pred_nb_scaled):.4f}")
print(f"Naive Bayes with scaling precision: {precision_score(y_val, y_pred_nb_scaled):.4f}")
print(f"Naive Bayes with scaling recall: {recall_score(y_val, y_pred_nb_scaled):.4f}")
print(f"Naive Bayes with scaling F1-score: {f1_score(y_val, y_pred_nb_scaled):.4f}")
print("Naive Bayes with scaling Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_nb_scaled))

Naive Bayes with scaling accuracy: 0.7523
Naive Bayes with scaling precision: 0.5003
Naive Bayes with scaling recall: 0.8510
Naive Bayes with scaling F1-score: 0.6301
Naive Bayes with scaling Confusion Matrix:
[[4897 1906]
 [ 334 1908]]


## 5. Decision Tree classifier

Decision Trees:
- split data recursively
- can easily overfit

### Tasks
- Train a decision tree
- Inspect tree depth
- Compute validation accuracy

**Reflection**
    - Why are trees easy to interpret?


In [47]:
# TODO: Train Decision Tree, prediction, depth and accuracy
# test multiple depths and compare results , save best accuracy
best_acc = 0
best_depth = 0

for depth in range(3, 9):
    dt = DecisionTreeClassifier(random_state=314, max_depth=depth)
    dt.fit(X_train, y_train)
    y_pred_dt = dt.predict(X_val)
    acc_score = accuracy_score(y_val, y_pred_dt)
    if acc_score > best_acc:
        best_acc = acc_score
        best_depth = depth
    print(f"Decision Tree (max_depth={depth}) accuracy: {acc_score:.4f}")
    print(f"Decision Tree (max_depth={depth}) precision: {precision_score(y_val, y_pred_dt):.4f}")
    print(f"Decision Tree (max_depth={depth}) recall: {recall_score(y_val, y_pred_dt):.4f}")
    print(f"Decision Tree (max_depth={depth}) F1-score: {f1_score(y_val, y_pred_dt):.4f}")
    print("Decision Tree Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred_dt))
    print(f"Decision Tree depth: {dt.get_depth()}")
    print("-" * 50)
    
print(f"Best Decision Tree accuracy: {best_acc:.4f}, for depth: {best_depth}")

Decision Tree (max_depth=3) accuracy: 0.8363
Decision Tree (max_depth=3) precision: 0.7342
Decision Tree (max_depth=3) recall: 0.5321
Decision Tree (max_depth=3) F1-score: 0.6170
Decision Tree Confusion Matrix:
[[6371  432]
 [1049 1193]]
Decision Tree depth: 3
--------------------------------------------------
Decision Tree (max_depth=4) accuracy: 0.8359
Decision Tree (max_depth=4) precision: 0.7300
Decision Tree (max_depth=4) recall: 0.5366
Decision Tree (max_depth=4) F1-score: 0.6185
Decision Tree Confusion Matrix:
[[6358  445]
 [1039 1203]]
Decision Tree depth: 4
--------------------------------------------------
Decision Tree (max_depth=5) accuracy: 0.8447
Decision Tree (max_depth=5) precision: 0.7705
Decision Tree (max_depth=5) recall: 0.5317
Decision Tree (max_depth=5) F1-score: 0.6292
Decision Tree Confusion Matrix:
[[6448  355]
 [1050 1192]]
Decision Tree depth: 5
--------------------------------------------------
Decision Tree (max_depth=6) accuracy: 0.8489
Decision Tree (max_

### 5a. Visualize the decision tree

Visualize in https://edotor.net/

In [45]:
from sklearn.tree import export_graphviz

dt = DecisionTreeClassifier(random_state=314, max_depth=3)
dt.fit(X_train, y_train)

print(export_graphviz(dt, out_file=None, feature_names=X.columns, filled=True))

digraph Tree {
node [shape=box, style="filled", color="black", fontname="helvetica"] ;
edge [fontname="helvetica"] ;
0 [label="marital-status=Married-civ-spouse <= 0.5\ngini = 0.373\nsamples = 36177\nvalue = [27211, 8966]", fillcolor="#eeab7a"] ;
1 [label="capital-gain <= 7055.5\ngini = 0.124\nsamples = 19291\nvalue = [18011, 1280]", fillcolor="#e78a47"] ;
0 -> 1 [labeldistance=2.5, labelangle=45, headlabel="True"] ;
2 [label="educational-num <= 13.5\ngini = 0.095\nsamples = 18940\nvalue = [17997, 943]", fillcolor="#e68843"] ;
1 -> 2 ;
3 [label="gini = 0.072\nsamples = 17889\nvalue = [17219, 670]", fillcolor="#e68641"] ;
2 -> 3 ;
4 [label="gini = 0.385\nsamples = 1051\nvalue = [778, 273]", fillcolor="#eead7e"] ;
2 -> 4 ;
5 [label="capital-gain <= 8296.0\ngini = 0.077\nsamples = 351\nvalue = [14, 337]", fillcolor="#41a1e6"] ;
1 -> 5 ;
6 [label="gini = 0.48\nsamples = 25\nvalue = [10, 15]", fillcolor="#bddef6"] ;
5 -> 6 ;
7 [label="gini = 0.024\nsamples = 326\nvalue = [4, 322]", fillcolo

## 6. Linear Support Vector Machine

Linear SVM:
- focuses on the decision boundary
- maximises the margin

### Tasks
- Train a Linear SVM
- Compute validation accuracy

**Reflection**
- How is this different from logistic regression?


In [54]:
# TODO: Train Linear SVM, prediction and accuracy
# Scale features before training SVM
scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train)
X_val_scaled = scaler2.transform(X_val)
svm = LinearSVC(random_state=314, max_iter=10000)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_val_scaled)
print(f"Linear SVM accuracy: {accuracy_score(y_val, y_pred_svm):.4f}")
print(f"Linear SVM precision: {precision_score(y_val, y_pred_svm):.4f}")
print(f"Linear SVM recall: {recall_score(y_val, y_pred_svm):.4f}")
print(f"Linear SVM F1-score: {f1_score(y_val, y_pred_svm):.4f}")
print("Linear SVM Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_svm))  


Linear SVM accuracy: 0.8469
Linear SVM precision: 0.7345
Linear SVM recall: 0.5986
Linear SVM F1-score: 0.6596
Linear SVM Confusion Matrix:
[[6318  485]
 [ 900 1342]]


## 7. Model comparison

### Tasks
- Compare validation performance across models
- Create a small summary table

**Reflection**
- Which model performs best?
- Is that the most important criterion?


In [50]:
# TODO: comparison table with all metrics for all models
results = {
    "Model": ["Naive Bayes", "Naive Bayes (scaled)", "Decision Tree (best)", "Linear SVM"],
    "Accuracy": [accuracy_score(y_val, y_pred_nb), accuracy_score(y_val, y_pred_nb_scaled), best_acc, accuracy_score(y_val, y_pred_svm)],
    "Precision": [precision_score(y_val, y_pred_nb), precision_score(y_val, y_pred_nb_scaled), precision_score(y_val, dt.predict(X_val)), precision_score(y_val, y_pred_svm)],
    "Recall": [recall_score(y_val, y_pred_nb), recall_score(y_val, y_pred_nb_scaled), recall_score(y_val, dt.predict(X_val)), recall_score(y_val, y_pred_svm)],
    "F1-score": [f1_score(y_val, y_pred_nb), f1_score(y_val, y_pred_nb_scaled), f1_score(y_val, dt.predict(X_val)), f1_score(y_val, y_pred_svm)]
}
results_df = pd.DataFrame(results)
print(results_df)


                  Model  Accuracy  Precision    Recall  F1-score
0           Naive Bayes  0.793809   0.674376  0.325156  0.438760
1  Naive Bayes (scaled)  0.752349   0.500262  0.851026  0.630119
2  Decision Tree (best)  0.851520   0.777709  0.560214  0.651283
3            Linear SVM  0.846877   0.734537  0.598573  0.659622


## 8. Critical reflection

Answer in Markdown:
- How do model assumptions affect results?
- Which model would you choose in practice, and why?
- What risks arise when models are used on human data?


### Reflection

(Write your answers here.)
